# CPU vs GPU: SLSQP single-shooting vs IPOPT collocation

Benchmarks the **calibration** step of `full_workflow_example.ipynb` across a 2x3 grid:

|              | SLSQP single-shooting | collocation (Gauss-Newton) | collocation (exact Hessian) |
|--------------|-----------------------|----------------------------|-----------------------------|
| **CPU**      | reference             |                            |                             |
| **GPU**      |                       |                            |                             |

**Why this comparison is interesting.** The two methods parallelise differently:

* **Single-shooting** rolls the trajectory forward *sequentially* — step `t+1` needs step `t`.
  It cannot be parallelised over time on any hardware, and on a GPU its per-step kernels are
  small enough to be launch-latency bound. Expect little or no speedup.
* **Collocation** evaluates all 360 segment defects and their derivatives *independently*, as
  one `vmap`. That is exactly the shape a GPU rewards.

So a GPU should widen the gap in collocation's favour. The question this notebook answers is
whether it widens it *enough* to overturn the CPU result.

**CPU baseline measured on this problem** (5-day window, 20-min steps, 28 parameters,
4 sensors; `pooled` = sum over sensors of `(RMSE/sd)^2`, the quantity every method minimises):

| arm | wall-clock | pooled |
|---|---|---|
| SLSQP single-shooting | 384 s | 30.4 |
| collocation, Gauss-Newton (300 it) | 409 s | 35.3 |
| collocation, exact Hessian (300 it) | 1480 s | 31.7 |

and the cost split that governs any GPU gain:

| | torch (offloadable) | IPOPT KKT (CPU-bound) | Amdahl ceiling |
|---|---|---|---|
| Gauss-Newton | 72.6 % | 27.4 % | 3.6x |
| exact Hessian | **95.4 %** | 4.6 % | **21.8x** |

The **irreducible** cost is ~0.227 s per IPOPT iteration, which no GPU touches. Gauss-Newton
needs ~2000 iterations to reach SLSQP-quality => ~454 s of pure CPU time, already more than
SLSQP's 384 s. So on GPU the *exact Hessian* is the only collocation variant that can win,
even though on CPU it is the slower of the two. This notebook tests that prediction.

**Runtime:** roughly 30-60 min for the full grid on CPU; the GPU arms are skipped
automatically when CUDA is unavailable.

In [ ]:
# --- Setup (Colab-aware) ---------------------------------------------------
# This notebook needs estimator features that are NOT on every branch:
#   * x0=None  (take a parameter's start value from initialize())
#   * data_warmstart / exact_hessian collocation options
# Point TWIN4BUILD_REF at a ref that has them.  Switch it to "dev" once the
# branch below is merged.
TWIN4BUILD_REF = "fix/issue-damper-ventilation-identifiability"

try:
    import twin4build as tb
except ImportError:
    # subprocess rather than %pip so the ref interpolates unambiguously.
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         f"git+https://github.com/JBjoernskov/Twin4Build.git@{TWIN4BUILD_REF}"],
        check=True,
    )
    import twin4build as tb

# Fail fast and loudly rather than 40 minutes into the grid: an older install
# raises a confusing "Initial value (x0) cannot be None" deep inside estimate().
import inspect

import twin4build.estimator._transcription as _tr

_src = inspect.getsource(_tr)
_missing = [n for n in ("data_warmstart", "exact_hessian") if n not in _src]
if not hasattr(tb.Estimator, "_resolve_x0"):
    _missing.append("x0=None support")
if _missing:
    raise RuntimeError(
        "The installed twin4build is missing: " + ", ".join(_missing)
        + f'''
Installed at: {tb.__file__}
Install a ref that has these, then restart the runtime:
    pip install -q --force-reinstall --no-deps git+https://github.com/JBjoernskov/Twin4Build.git@{TWIN4BUILD_REF}
    (Colab: Runtime > Restart session, then re-run this cell)'''
    )
print(f"twin4build OK at {tb.__file__}")

import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

DEVICES = ["cpu"] + (["cuda"] if torch.cuda.is_available() else [])
print(f"torch {torch.__version__}")
print(f"CPU threads: {torch.get_num_threads()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}  ({props.total_memory / 1e9:.1f} GB, sm_{props.major}{props.minor})")
    # fp64 throughput is the gating factor -- see the FP32 section at the end.
    print("NOTE: collocation needs float64 (constr_viol_tol=1e-8 is below fp32 precision).")
    print("      Consumer cards run fp64 at 1/32-1/64 of fp32; datacenter cards at 1/2.")
else:
    print("No GPU available -- only the CPU arms will run.")
    print("(In Colab: Runtime > Change runtime type > GPU)")

## Part A — The model

Same topology and settings as `full_workflow_example.ipynb`, with two differences that make
this runnable anywhere and reproducible:

1. **Sensor CSVs are repointed to the packaged copies** under `twin4build/examples/estimator_example/`,
   so the notebook does not depend on a local checkout of the case-study repo. Column indices
   are unchanged (these are the same files).
2. **The parameter list is pinned here** rather than imported, so the benchmark does not
   silently change if the example is edited.

The occupancy-detector `threshold` **is estimated**. It has to be: the gate fires on
`N_occ > threshold`, and `N_occ` scales as `1/G_occ`, which is itself estimated — so a
hard-coded threshold makes the ventilation branch a degenerate direction of the objective and
the damper prediction collapses. Threshold and `G_occ` are identifiable only as a pair.

In [ ]:
import datetime
from dateutil import tz

import twin4build.examples.utils as utils
import twin4build.utils.types as tps
from twin4build.examples.full_workflow_example import fcn as _example_fcn
from twin4build.utils.rgetattr import rgetattr

STEP = 1200  # 20 minutes
START = [datetime.datetime(2023, 12, 2, tzinfo=tz.gettz("Europe/Copenhagen"))]
END = [datetime.datetime(2023, 12, 7, tzinfo=tz.gettz("Europe/Copenhagen"))]
N_WARMUP = 20


def _repoint_to_packaged_csvs(self):
    '''Use the CSVs shipped inside the package instead of a local case-study path.

    Same files, so the (datecolumn=2, valuecolumn=4) layout the example sets is kept.
    '''
    P = lambda name: utils.get_path(["estimator_example", name])
    for comp_id, name in [
        ("office_temperature_sensor", "temperature_sensor.csv"),
        ("office_co2_sensor", "co2_sensor.csv"),
        ("office_valve_position_sensor", "valve_position_sensor.csv"),
        ("office_damper_position_sensor", "damper_position_sensor.csv"),
        ("supply_air_temperature_sensor", "supply_air_temperature.csv"),
        ("office_temperature_heating_setpoint", "temperature_heating_setpoint.csv"),
    ]:
        self.components[comp_id].filename = P(name)
    occ = self.components["office_occupancy"]
    occ.co2_filename = P("co2_sensor.csv")
    occ.damper_filename = P("damper_position_sensor.csv")


def fcn(self):
    _example_fcn(self)              # topology + calibrated settings from the example
    _repoint_to_packaged_csvs(self)  # ...but portable data paths


def build_model(device="cpu", dtype=torch.float64):
    model = tb.Model(id=f"bench_{device}")
    model.load(
        semantic_model_filename=utils.get_path(
            ["estimator_example", "one_room_example_model.xlsm"]
        ),
        fcn=fcn,
    )
    model.to(device, dtype)
    return model


def build_parameters(model):
    '''Pinned copy of the example's estimation problem (28 scalar parameters).'''
    space = model.components["office"]
    heater = model.components["office_space_heater"]
    hc = model.components["office_temperature_heating_controller"]
    cc = model.components["office_co2_controller"]
    valve = model.components["office_space_heater_valve"]
    sup = model.components["office_supply_damper"]
    exh = model.components["office_exhaust_damper"]
    occ = model.components["office_occupancy"]
    wall = model.components["office_boundary_wall"]
    det = model.components["office_occupancy_detector"]
    return [
        (space, "thermal.C_air", 5e5, 1e4, 5e5),
        (space, "thermal.C_wall", 1e6, 1e5, 3e6),
        (wall, "C", 1e6, 1e4, 1e7),
        (space, "thermal.R_out", 0.5, 0.01, 1),
        (space, "thermal.R_in", 0.1, 0.01, 1),
        (wall, "R_a", 0.04, 1e-4, 1),
        (wall, "R_b", 0.04, 1e-4, 1),
        (space, "thermal.f_wall", 0.1, 0, 10),
        (space, "thermal.f_air", 0.1, 0, 10),
        (space, "thermal.Q_occ_gain", 100.0, 10, 200),
        (heater, "thermalMassHeatCapacity", 1e4, 1e3, 2e5),
        (heater, "UA", None, 1, 100),  # x0 from initialize_UA
        (hc, "kp", 0.005, 1e-5, 1, "private"),
        (cc, "kp", 0.0001, 1e-5, 1, "private"),
        ([hc, cc], "Ti", 30, 1, 300, "private"),
        ([hc, cc], "Td", 0, 0, 1, "private"),
        (valve, "waterFlowRateMax", 0.001, 1e-6, 0.1),
        (valve, "valveAuthority", 1, 0.4, 1),
        ([sup, occ.supply_damper], "a", 1, 1, 10, "shared"),
        ([sup, occ.supply_damper], "nominalAirFlowRate", 0.1, 1e-5, 1, "shared"),
        ([exh, occ.exhaust_damper], "a", 1, 1, 10, "shared"),
        ([exh, occ.exhaust_damper], "nominalAirFlowRate", 0.1, 1e-5, 1, "shared"),
        ([space, occ], "mass.V", 65, 50, 80, "shared"),
        ([space, occ], "mass.G_occ", 1e-6, 1e-6, 1e-5, "shared"),
        ([space, occ], "mass.m_inf", 0.001, 1e-4, 0.01, "shared"),
        # Identifiable only jointly with G_occ -- see the markdown above.
        (det, "threshold", 1.0, 0.02, 5.0),
    ]


def build_measurements(model):
    return [
        (model.components["office_valve_position_sensor"], 0.05 / 2),
        (model.components["office_temperature_sensor"], 0.1 / 2),
        (model.components["office_damper_position_sensor"], 0.05 / 2),
        (model.components["office_co2_sensor"], 30 / 2),
    ]


SD = {  # the weights above, for scoring
    "office_temperature_sensor": 0.05,
    "office_valve_position_sensor": 0.025,
    "office_damper_position_sensor": 0.025,
    "office_co2_sensor": 15.0,
}
print("model builders ready")

## Part B — The 2x3 benchmark

Every arm starts from the **same** 5-iteration SLSQP warm start, so they are compared from an
identical point. Then:

* **SLSQP** continues single-shooting to convergence (`maxiter=100`).
* **collocation** hands that point to IPOPT with `data_warmstart=False` — essential, because
  the default seeds boundary states from *measurements*, which violates the continuity defects
  and throws the warm start away before IPOPT's first step.

`exact_hessian=True` adds the constraint curvature `sum(lam*d2g)` and the residual curvature
`sum(r*d2r)` that plain Gauss-Newton drops. On CPU it costs ~3.6x per iteration; the point of
this benchmark is that on GPU that cost is parallel work while its benefit — fewer *serial*
IPOPT factorisations — is not.

Scoring is always a **real `do_step` simulation** of the returned parameters, never the NLP's
internal view, so the numbers are comparable across methods.

In [ ]:
WARM_ITERS = 5
SLSQP_ITERS = 100
COLLOC_ITERS = 300


def score(model):
    '''Per-sensor RMSE + pooled objective from a real simulation.'''
    out, pooled = {}, 0.0
    for cid, sd in SD.items():
        c = model.components[cid]
        sim = c.output["measuredValue"].history()[:, 0, 0].detach().cpu().numpy()[N_WARMUP:]
        act = c.time_series_input.values[:, 0, 0].detach().cpu().numpy()[N_WARMUP:]
        r = float(np.sqrt(np.mean((sim - act) ** 2)))
        out[cid.replace("office_", "").replace("_sensor", "")] = r
        pooled += (r / sd) ** 2
    out["pooled"] = pooled
    d = model.components["office_damper_position_sensor"]
    ds = d.output["measuredValue"].history()[:, 0, 0].detach().cpu().numpy()[N_WARMUP:]
    da = d.time_series_input.values[:, 0, 0].detach().cpu().numpy()[N_WARMUP:]
    on_a, on_s = da > 0.15, ds > 0.15
    out["damper_FP"] = float(np.mean(~on_a & on_s))
    out["damper_FN"] = float(np.mean(on_a & ~on_s))
    return out


def _carry_x0(entry, model):
    '''Re-read each parameter group's fitted value off the model.

    estimate() reorders parameters internally (private first, then shared), so
    result["result_x"] is NOT aligned with the input list -- reading the model is
    the supported way to chain one estimate into the next.
    '''
    comps, attr, _x0, lo, hi = entry[:5]
    comp = comps[0] if isinstance(comps, list) else comps
    v = float(rgetattr(comp, attr).get().reshape(-1)[0])
    eps = 1e-9 * (hi - lo)
    return (comps, attr, min(max(v, lo + eps), hi - eps), lo, hi, *entry[5:])


def run_arm(device, method, dtype=torch.float64):
    model = build_model(device, dtype)
    sim = tb.Simulator(model)
    est = tb.Estimator(sim)
    params = build_parameters(model)
    meas = build_measurements(model)

    t0 = time.perf_counter()
    est.estimate(START, END, STEP, params, meas, n_warmup=N_WARMUP,
                 method=("scipy", "SLSQP", "ad"),
                 options={"maxiter": WARM_ITERS, "fast": True})
    params2 = [_carry_x0(e, model) for e in params]

    audit = None
    if method == "slsqp":
        est.estimate(START, END, STEP, params2, meas, n_warmup=N_WARMUP,
                     method=("scipy", "SLSQP", "ad"),
                     options={"maxiter": SLSQP_ITERS, "fast": True})
        seed = {}
    else:
        opts = {"maxiter": COLLOC_ITERS, "data_warmstart": False,
                "early_stopping": False,
                "exact_hessian": method == "collocation_exact"}
        r = est.estimate(START, END, STEP, params2, meas, n_warmup=N_WARMUP,
                         method=("casadi", "ipopt", "ad", "collocation"), options=opts)
        seed = r.get("estimated_initial_state", {}) or {}
        audit = r.get("transcription_audit")
    wall = time.perf_counter() - t0

    def _seed():
        for cid, x0 in seed.items():
            model.get_component(cid).set_state(x0)

    model.set_save_simulation_result(flag=True)
    sim.simulate(step_size=STEP, start_time=START, end_time=END, after_initialize=_seed)

    row = {"device": device, "method": method, "seconds": wall}
    row.update(score(model))
    if audit:
        row["max_defect"] = float(audit["max_defect"])
    return row


METHODS = ["slsqp", "collocation_gn", "collocation_exact"]
rows = []
for dev in DEVICES:
    for m in METHODS:
        print(f"\n=== {dev:4s} | {m} ===", flush=True)
        try:
            rows.append(run_arm(dev, m))
            print(f"    {rows[-1]['seconds']:.0f}s  pooled={rows[-1]['pooled']:.2f}", flush=True)
        except Exception as exc:  # keep the grid going if one arm fails
            print(f"    FAILED: {type(exc).__name__}: {exc}", flush=True)
            rows.append({"device": dev, "method": m, "seconds": np.nan, "pooled": np.nan,
                         "error": f"{type(exc).__name__}: {exc}"})

results = pd.DataFrame(rows)

# Do NOT let a failed arm hide in a DataFrame column -- surface it immediately.
if "error" in results.columns and results["error"].notna().any():
    print()
    print("!!! SOME ARMS FAILED -- the table below is incomplete !!!")
    for _, r in results[results["error"].notna()].iterrows():
        print(f"    {r['device']:4s} {r['method']:20s} {r['error']}")
    print("If this says 'Initial value (x0) cannot be None', the installed")
    print("twin4build predates this notebook -- see the setup cell.")
    print()

results

## Part C — Where the time goes (the Amdahl ceiling)

A GPU can only accelerate the **torch** callbacks (objective, gradient, constraints, constraint
Jacobian, Hessian). IPOPT's sparse KKT factorisation stays on the CPU. The fraction of
wall-clock spent in torch is therefore a hard ceiling on any GPU speedup — and the *remainder*,
multiplied by the number of iterations a method needs, is a floor no hardware removes.

This is the measurement that decides whether Gauss-Newton or the exact Hessian is the right
variant to put on a GPU.

In [ ]:
import functools

import twin4build.estimator._casadi_ipopt as _ipopt


def profile_split(device, exact, maxiter=30, dtype=torch.float64):
    T = {k: 0.0 for k in ("obj", "grad", "g", "gjac", "hess")}

    def timed(fn, key):
        if fn is None:
            return None

        @functools.wraps(fn)  # keeps signature visible: solve_ipopt_constrained
        def w(*a, **kw):      # probes hess_vals arity to decide on lam_g
            t0 = time.perf_counter()
            try:
                return fn(*a, **kw)
            finally:
                T[key] += time.perf_counter() - t0

        return w

    model = build_model(device, dtype)
    sim = tb.Simulator(model)
    est = tb.Estimator(sim)
    orig = _ipopt.solve_ipopt_constrained
    box = {}

    def wrapped(x0, lb, ub, fun, grad, n_g, g_fun, g_jac_vals, jr, jc,
                options=None, *, hess_vals=None, **kw):
        t0 = time.perf_counter()
        try:
            return orig(x0, lb, ub, timed(fun, "obj"), timed(grad, "grad"), n_g,
                        timed(g_fun, "g"), timed(g_jac_vals, "gjac"), jr, jc,
                        options, hess_vals=timed(hess_vals, "hess"), **kw)
        finally:
            box["total"] = time.perf_counter() - t0

    _ipopt.solve_ipopt_constrained = wrapped
    try:
        est.estimate(START, END, STEP, build_parameters(model), build_measurements(model),
                     n_warmup=N_WARMUP, method=("casadi", "ipopt", "ad", "collocation"),
                     options={"maxiter": maxiter, "data_warmstart": False,
                              "early_stopping": False, "exact_hessian": exact})
    finally:
        _ipopt.solve_ipopt_constrained = orig

    total = box["total"]
    torch_s = sum(T.values())
    return {
        "device": device,
        "variant": "exact" if exact else "gauss-newton",
        "torch_frac": torch_s / total,
        "ipopt_frac": (total - torch_s) / total,
        "s_per_iter_ipopt": (total - torch_s) / maxiter,
        "amdahl_ceiling": 1.0 / max(1e-9, 1 - torch_s / total),
        **{f"{k}_s": v for k, v in T.items()},
    }


splits = pd.DataFrame([
    profile_split(dev, exact)
    for dev in DEVICES
    for exact in (False, True)
])
splits[["device", "variant", "torch_frac", "ipopt_frac", "s_per_iter_ipopt", "amdahl_ceiling"]]

## Part D — Does float32 work at all?

This is the gating risk for consumer GPUs, where fp64 runs at 1/32–1/64 of fp32 throughput.

Collocation sets `constr_viol_tol = 1e-8`, but float32 carries only ~1e-7 relative precision —
so the continuity defects may be **unable** to converge in single precision no matter how many
iterations you allow. If `max|defect|` below stalls well above 1e-8, fp32 is not a usable
shortcut for collocation and the GPU question reduces to whether you have fp64-capable hardware.

(The forward-simulation and optimisation benchmarks in the sibling notebooks *do* benefit from
fp32 — this limitation is specific to collocation's hard equality constraints.)

In [ ]:
fp32_rows = []
for dev in DEVICES:
    for dt, label in [(torch.float64, "float64"), (torch.float32, "float32")]:
        try:
            model = build_model(dev, dt)
            sim = tb.Simulator(model)
            est = tb.Estimator(sim)
            r = est.estimate(START, END, STEP, build_parameters(model),
                             build_measurements(model), n_warmup=N_WARMUP,
                             method=("casadi", "ipopt", "ad", "collocation"),
                             options={"maxiter": 60, "data_warmstart": False,
                                      "early_stopping": False, "exact_hessian": True})
            a = r.get("transcription_audit") or {}
            fp32_rows.append({"device": dev, "dtype": label,
                              "max_defect": a.get("max_defect", np.nan),
                              "feasible_at_1e-8": bool(a.get("max_defect", 1) < 1e-8)})
        except Exception as exc:
            fp32_rows.append({"device": dev, "dtype": label, "max_defect": np.nan,
                              "feasible_at_1e-8": False, "error": str(exc)[:80]})
        finally:
            tps.set_float_dtype(torch.float64)  # restore the process-global dtype

pd.DataFrame(fp32_rows)

In [ ]:
# --- Summary ---------------------------------------------------------------
ok = results.dropna(subset=["seconds"])
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

piv = ok.pivot(index="method", columns="device", values="seconds")
piv.plot.bar(ax=axes[0], rot=15)
axes[0].set_ylabel("wall-clock [s]")
axes[0].set_title("Time to calibrate (lower is better)")
axes[0].grid(axis="y", alpha=0.3)

for dev, g in ok.groupby("device"):
    axes[1].scatter(g["seconds"], g["pooled"], s=90, label=dev)
    for _, r in g.iterrows():
        axes[1].annotate(r["method"].replace("collocation_", "col-"),
                         (r["seconds"], r["pooled"]), fontsize=8,
                         xytext=(4, 4), textcoords="offset points")
axes[1].set_xlabel("wall-clock [s]")
axes[1].set_ylabel("pooled objective (lower is better)")
axes[1].set_title("Quality vs cost -- bottom-left wins")
axes[1].grid(alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.show()

print(ok[["device", "method", "seconds", "pooled", "temperature",
          "damper_position", "co2"]].to_string(index=False))

## How to read the results

**The comparison to make is quality-at-equal-time, not time-at-equal-iterations.** Collocation
solves a 5427-variable NLP where single-shooting solves a 28-variable one, so per-iteration
costs are not comparable; only the wall-clock to reach a given `pooled` is.

**What the CPU numbers already tell us:**

* Collocation *can* match single-shooting on accuracy here — an L-BFGS run reached pooled 29.72
  against SLSQP's 30.41. Earlier reports that it converges to a worse optimum were an artefact of
  premature termination, not a property of the method. In fact collocation's feasible set
  *contains* single-shooting's (any rollout is a feasible collocation point with the same
  objective), so a converged collocation must reach an equal-or-better objective — any worse
  result is a non-convergence certificate.
* It costs 4-10x the wall-clock to get there, because this model is **dissipative**: per-step
  Jacobians contract, so single-shooting's gradient was never ill-conditioned and collocation's
  central advantage is absent. Collocation earns its keep on stiff or unstable dynamics, which is
  where the literature applies it.

**What to look for on GPU:**

1. **Does SLSQP speed up?** Expect *no* — its rollout is sequential and its kernels are tiny.
   If it slows down, that is launch latency, and it is the expected result rather than a bug.
2. **Does the exact Hessian speed up more than Gauss-Newton?** It should: 95.4 % of its time is
   the `vmap(jacfwd(jacrev))` Hessian, versus 72.6 % for Gauss-Newton.
3. **Multiply the residual IPOPT fraction by the iteration count.** With ~0.227 s/iteration on
   CPU, Gauss-Newton's ~2000 iterations cost ~454 s of unavoidable CPU time — more than SLSQP's
   whole run. That is why Gauss-Newton cannot win on a GPU while the exact Hessian can, despite
   being the slower variant on CPU.
4. **Check Part D first.** If float32 cannot hold the defects, a consumer GPU's fp64 penalty may
   erase the advantage before any of the above matters.

**Caveat on the fit itself.** The two methods land on different points of a flat trade-off —
collocation tends to fit CO2 better, SLSQP the damper and temperature. `pooled` compresses that
into one number; check the per-sensor columns before concluding one dominates.